In [9]:
import os
import shutil
import random
from google.colab import drive

# ==========================================
# MOUNT DRIVE
# ==========================================
drive.mount('/content/drive')

# ==========================================
# PATHS
# ==========================================
drive_folder = "/content/drive/MyDrive/Nexar_event_batches_3DCNN"
ssd_folder = "/content/ssd_3dcnn"

if not os.path.exists(ssd_folder):
    os.makedirs(ssd_folder)

# ==========================================
# COPY TO SSD (only once)
# ==========================================
drive_files = sorted([
    f for f in os.listdir(drive_folder)
    if f.endswith(".npz")
])

print("Total batches found:", len(drive_files))

for f in drive_files:
    src = os.path.join(drive_folder, f)
    dst = os.path.join(ssd_folder, f)

    if not os.path.exists(dst):
        shutil.copy(src, dst)

print("✅ Copied to SSD")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total batches found: 8
✅ Copied to SSD


In [10]:
# ==========================================
# TRAIN / VAL / TEST SPLIT (BY CHUNK)
# ==========================================
import random

random.seed(42)
random.shuffle(drive_files)

total_chunks = len(drive_files)

test_ratio = 0.3
val_ratio = 0.2

test_count = int(total_chunks * test_ratio)
val_count = int(total_chunks * val_ratio)

test_files = drive_files[:test_count]
val_files = drive_files[test_count:test_count + val_count]
train_files = drive_files[test_count + val_count:]

print("Train chunks:", len(train_files))
print("Val chunks:", len(val_files))
print("Test chunks:", len(test_files))

Train chunks: 5
Val chunks: 1
Test chunks: 2


In [11]:
import numpy as np
import gc

def load_all_npz(file_list):
    X_list = []
    y_list = []

    for f in file_list:
        data = np.load(os.path.join(ssd_folder, f))
        X_list.append(data["X"])
        y_list.append(data["y"])

    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)

    return X.astype(np.float32), y.astype(np.float32)

# Load validation once
X_val, y_val = load_all_npz(val_files)

print("Validation shape:", X_val.shape)
print("Validation distribution:", np.unique(y_val, return_counts=True))

Validation shape: (200, 8, 224, 224, 3)
Validation distribution: (array([0., 1.], dtype=float32), array([106,  94]))


In [12]:
from tensorflow import keras
from tensorflow.keras import layers

input_shape = (8, 224, 224, 3)

inputs = keras.Input(shape=input_shape)

# Block 1
x = layers.Conv3D(32, (3,3,3), padding="same")(inputs)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.MaxPooling3D((1,2,2))(x)

# Block 2
x = layers.Conv3D(64, (3,3,3), padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.MaxPooling3D((2,2,2))(x)

# Block 3
x = layers.Conv3D(128, (3,3,3), padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.MaxPooling3D((2,2,2))(x)

# Block 4 (NEW)
x = layers.Conv3D(256, (3,3,3), padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.MaxPooling3D((2,2,2))(x)

x = layers.GlobalAveragePooling3D()(x)

x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)

x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.4)(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(3e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 8, 224, 224, 3) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_3 (Conv3D)               │ (None, 8, 224, 224,    │         2,624 │
│                                 │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 8, 224, 224,    │           128 │
│ (BatchNormalization)            │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 8, 224, 224,    │             0 │
│                                 │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_3 (MaxPooling3D)  │ (None, 8, 112, 112,    │             0 │
│                                 │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_4 (Conv3D)               │ (None, 8, 112, 112,    │        55,360 │
│                                 │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 8, 112, 112,    │           256 │
│ (BatchNormalization)            │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 8, 112, 112,    │             0 │
│                                 │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_4 (MaxPooling3D)  │ (None, 4, 56, 56, 64)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_5 (Conv3D)               │ (None, 4, 56, 56, 128) │       221,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 4, 56, 56, 128) │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 4, 56, 56, 128) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_5 (MaxPooling3D)  │ (None, 2, 28, 28, 128) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_6 (Conv3D)               │ (None, 2, 28, 28, 256) │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 2, 28, 28, 256) │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 2, 28, 28, 256) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_6 (MaxPooling3D)  │ (None, 1, 14, 14, 256) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling3d_1      │ (None, 256)            │             0 │
│ (GlobalAveragePooling3D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             

 Total params: 1,248,513 (4.76 MB)

 Trainable params: 1,247,553 (4.76 MB)

 Non-trainable params: 960 (3.75 KB)

In [13]:
import gc
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=2,
    verbose=1
)

callbacks = [early_stop, reduce_lr]

batch_size = 8
max_global_epochs = 10

for global_epoch in range(max_global_epochs):

    print(f"\n===== Global Epoch {global_epoch+1} =====")

    random.shuffle(train_files)

    for chunk_file in train_files:

        data = np.load(os.path.join(ssd_folder, chunk_file))
        X_chunk = data["X"].astype(np.float32)
        y_chunk = data["y"].astype(np.float32)

        print("Training on chunk:", chunk_file)
        print("Chunk shape:", X_chunk.shape)

        model.fit(
            X_chunk,
            y_chunk,
            validation_data=(X_val, y_val),
            batch_size=batch_size,
            epochs=1,
            callbacks=callbacks,
            verbose=1
        )

        del X_chunk
        del y_chunk
        del data
        gc.collect()

    if early_stop.stopped_epoch > 0:
        print("Early stopping triggered.")
        break


===== Global Epoch 1 =====
Training on chunk: lstm_batch_2.npz
Chunk shape: (200, 8, 224, 224, 3)
25/25 ━━━━━━━━━━━━━━━━━━━━ 33s 563ms/step - accuracy: 0.4408 - loss: 0.8511 - val_accuracy: 0.5300 - val_loss: 0.6907 - learning_rate: 3.0000e-05
Restoring model weights from the end of the best epoch: 1.
Training on chunk: lstm_batch_5.npz
Chunk shape: (200, 8, 224, 224, 3)
25/25 ━━━━━━━━━━━━━━━━━━━━ 12s 479ms/step - accuracy: 0.6099 - loss: 0.7095 - val_accuracy: 0.5300 - val_loss: 0.6989 - learning_rate: 3.0000e-05
Restoring model weights from the end of the best epoch: 1.
Training on chunk: lstm_batch_0.npz
Chunk shape: (200, 8, 224, 224, 3)
25/25 ━━━━━━━━━━━━━━━━━━━━ 12s 501ms/step - accuracy: 0.5449 - loss: 0.7337 - val_accuracy: 0.5300 - val_loss: 0.7213 - learning_rate: 3.0000e-05
Restoring model weights from the end of the best epoch: 1.
Training on chunk: lstm_batch_1.npz
Chunk shape: (200, 8, 224, 224, 3)
25/25 ━━━━━━━━━━━━━━━━━━━━ 12s 494ms/step - accuracy: 0.6594 - loss: 0.64

In [14]:
X_test, y_test = load_all_npz(test_files)

In [15]:
print("\n===== FINAL TEST EVALUATION =====")
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)


===== FINAL TEST EVALUATION =====
13/13 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.7291 - loss: 0.5319
Test Loss: 0.5140286684036255
Test Accuracy: 0.7400000095367432
